# Ozon Performance — demo

Notebook для работы с функциями из `ozon_performance.py`:

1. `get_campaign_dict()` — справочник кампаний
2. `get_campaigns_daily_stat(date_from, date_to)` — дневная статистика кампаний
3. `get_ads_daily_stat(date_from, date_to)` — дневная статистика объявлений
4. `get_video_ads_daily_stat(date_from, date_to)` — дневная статистика видео-объявлений
5. `get_reach_campaigns_daily_stat(global_start_date, date_from, date_to)` — охват кампаний (кумулятив)
6. `get_reach_ads_daily_stat(global_start_date, date_from, date_to)` — охват объявлений (кумулятив)

## Перед запуском
1. Установи зависимости (ячейка ниже)
2. Убедись, что `.env` содержит `CLIENT_ID` и `CLIENT_SECRET`

## 1. Установка зависимостей (один раз)

In [ ]:
%pip install -q -r requirements.txt

## 2. Импорты и загрузка учётных данных

In [ ]:
import logging
import os
from dotenv import load_dotenv

load_dotenv(".env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

assert os.environ.get("CLIENT_ID"), "CLIENT_ID отсутствует в .env"
assert os.environ.get("CLIENT_SECRET"), "CLIENT_SECRET отсутствует в .env"

print("Учётные данные загружены")

In [ ]:
import sys
sys.path.insert(0, "..")

from ozon_performance.ozon_performance import (
    get_campaign_dict,
    get_campaigns_daily_stat,
    get_ads_daily_stat,
    get_video_ads_daily_stat,
    get_reach_campaigns_daily_stat,
    get_reach_ads_daily_stat,
)

## 3. Параметры периода

In [ ]:
from datetime import date, timedelta

DATE_TO   = (date.today() - timedelta(days=1)).isoformat()
DATE_FROM = (date.today() - timedelta(days=7)).isoformat()

# Переопредели вручную если нужно:
# DATE_FROM = "2026-01-01"
# DATE_TO   = "2026-01-07"

# Начало накопительного периода для охвата (должно быть <= DATE_FROM)
GLOBAL_START_DATE = DATE_FROM
# GLOBAL_START_DATE = "2026-01-01"

print(f"Период: {DATE_FROM} → {DATE_TO}")
print(f"Глобальный старт (охват): {GLOBAL_START_DATE}")

## 4. Справочник кампаний

In [ ]:
campaign_dict = get_campaign_dict()
print(f"Строк: {len(campaign_dict)}, колонки: {list(campaign_dict.columns)}")
campaign_dict.head(10)

In [ ]:
if not campaign_dict.empty:
    print(f"Всего кампаний: {len(campaign_dict)}")

## 5. Статистика кампаний по дням

In [ ]:
campaigns_stat = get_campaigns_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(campaigns_stat)}, колонки: {list(campaigns_stat.columns)}")
campaigns_stat.head(10)

In [ ]:
if not campaigns_stat.empty:
    summary = campaigns_stat.groupby("date")[["views", "clicks", "costs_nds"]].sum()
    summary

## 6. Статистика объявлений по дням

In [ ]:
ads_stat = get_ads_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(ads_stat)}, колонки: {list(ads_stat.columns)}")
ads_stat.head(10)

In [ ]:
if not ads_stat.empty:
    summary = ads_stat.groupby("date")[["views", "clicks", "costs_nds"]].sum()
    summary

## 7. Статистика видео-объявлений по дням

In [ ]:
video_stat = get_video_ads_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(video_stat)}, колонки: {list(video_stat.columns)}")
video_stat.head(10)

In [ ]:
if not video_stat.empty:
    summary = video_stat.groupby("date")[["views", "clicks", "costs_nds"]].sum()
    summary

## 8. Охват кампаний

> **Важно:** `reach` — кумулятивный показатель. Для каждого дня D отправляется запрос
> за период [GLOBAL_START_DATE, D] с groupBy=NO_GROUP_BY.
> Количество API-запросов = N_дней × N_батчей.
> `GLOBAL_START_DATE` должна быть ≤ `DATE_FROM`.

In [ ]:
reach_campaigns = get_reach_campaigns_daily_stat(GLOBAL_START_DATE, DATE_FROM, DATE_TO)
print(f"Строк: {len(reach_campaigns)}, колонки: {list(reach_campaigns.columns)}")
reach_campaigns.head(10)

In [ ]:
if not reach_campaigns.empty:
    top = (
        reach_campaigns[reach_campaigns["date"] == DATE_TO]
        .sort_values("reach", ascending=False)
        .head(10)
    )
    top

## 9. Охват объявлений

> **Важно:** аналогично разделу 8 — кумулятивный показатель, запросы за [GLOBAL_START_DATE, D].

In [ ]:
reach_ads = get_reach_ads_daily_stat(GLOBAL_START_DATE, DATE_FROM, DATE_TO)
print(f"Строк: {len(reach_ads)}, колонки: {list(reach_ads.columns)}")
reach_ads.head(10)

In [ ]:
if not reach_ads.empty:
    top = (
        reach_ads[reach_ads["date"] == DATE_TO]
        .sort_values("reach", ascending=False)
        .head(10)
    )
    top

## Сохранение в CSV (опционально)

In [ ]:
today = date.today().isoformat()

campaign_dict.to_csv(f"campaign_dict_{today}.csv", index=False, encoding="cp1251", errors="replace")
campaigns_stat.to_csv(f"campaigns_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
ads_stat.to_csv(f"ads_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
video_stat.to_csv(f"video_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
reach_campaigns.to_csv(f"reach_campaigns_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
reach_ads.to_csv(f"reach_ads_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")

print("Файлы сохранены")